# Forecasting case-study: Bike riding in Paris

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/06_learning_with_time_series/04c-forecasting_case_study.ipynb)

```
Authors: Thomas Moreau
         Pedro Rodrigues
         Mathurin Massias
         Alexandre Gramfort
```



### Overview

In the recent years, bikes have been used in much larger proportion to move world-wide, and in particular in Paris. They account for 15% of the every day travel inside the city and the number of bike travel increased by 70% in the last year. Today, there exists great interest in studying the bike flux to their important role in traffic, environmental and health issues. In particular, it is an important information for the city council to know the number of bikes that will travel on different axis in the next days, weeks or months to plan the city's infrastructure.

### Attribute Information

The dataset was collected with cyclist counters installed by Paris city council in multiple locations. It contains hourly information about cyclist traffic, as well as the following features:

- counter name
- counter site name
- date
- counter installation date
- latitude and longitude

Available features are quite scarce. However, we can also use any **external data** that can help you to predict the target variable.

### Objective: forecast the number of bike rides

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

We will use the parquet format. We do not use a .csv file as parquet files are a lot faster to load and it produces much smaller files. It also allows to store the proper dtype of each column. You should consider it for your work!

In [ ]:
DATA_FILE = 'data/bike-counter-data.parquet'
URL_REPO = "https://github.com/x-datascience-datacamp/datacamp-master/raw/main/06_learning_with_time_series/"

if Path(DATA_FILE).exists():
    df = pd.read_parquet(DATA_FILE)
else:
    df = pd.read_parquet(f"{URL_REPO}{DATA_FILE}")
df.head()

#### Let's first look at the target one wants to predict

One can see that the target is quite heavy tailed

In [ ]:
import seaborn as sns
ax = sns.histplot(df, x='bike_count', kde=True, bins=50)

## Exploratory Data Analysis (EDA)

In [ ]:
df.info()

In [ ]:
df.nunique(axis=0)

In [ ]:
df.describe()

Let's look at the most used sites. Note that one can have 2 counters in one site.

In [ ]:
df.groupby(['site_name', 'counter_name'], observed=True)['bike_count'].sum().sort_values(
    ascending=False
).head(10).to_frame()

In [ ]:
import folium

m = folium.Map(location=df[['latitude', 'longitude']].mean(axis=0).values, zoom_start=13)

for _, row in df[['counter_name', 'latitude', 'longitude']].drop_duplicates('counter_name').iterrows():
    folium.Marker(
    row[['latitude', 'longitude']].values.tolist(), popup=row['counter_name']
    ).add_to(m)

m

Next we will look into the temporal distribution of the most frequently used bike counters. If we plot it directly we will not see much because there are half a million data points.

We select the rows of one counter with `DataFrame.query`, which filters rows with a condition written as a string.

In [ ]:
query_str = "counter_name == 'Totem 73 boulevard de Sébastopol S-N'"
_ = df.query(query_str).sort_values('date').plot(x='date', y='bike_count')

Instead we aggregate the data, for instance, by week to have a clearer overall picture.

In [ ]:
_ = df.query(query_str).groupby(pd.Grouper(freq='1W', key='date'))['bike_count'].mean().plot()

While at the same time, we can zoom on a week in particular for a more short-term visualization:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.query(query_str + " and '2021/03/30' < date < '2021/04/05'").sort_values('date').plot(x='date', y='bike_count', ax=ax)

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Plot the average bike_count per hour of the week (averaged over all sites).</li>
    </ul>
</div>

You should clearly distinguish the commute patterns in the morning and evenings of the work days
and the leisure use of the bikes on the weekends with a more spread peak demand around the middle of the days.

Solution is in `solutions/bike-0_averaging-per-weekday-hour.py`

## Metrics

There are many possible metrics for regression tasks like here. In what follows we will be using the RMSLE, that penalizes less large deviations than the MLE.

#### Root Mean Squared Logarithmic Error (RMSLE)

RMSLE = $\sqrt{ \frac{1}{N} \sum_{i=1}^N (\log(x_i + 1) - \log(y_i + 1))^2 }$

In what follows we will be using the RMSLE, that penalizes less large deviations than the MSE. For this we will be creating a new column called `log_bike_count` using the `np.log1p` function.

In [ ]:
import numpy as np
df['log_bike_count'] = np.log1p(df['bike_count'])

## Train and test data

We will now focus on one site and focus on the forecast of its bike count.

In [ ]:
counter_name = 'Totem 73 boulevard de Sébastopol S-N'

data = df.query("counter_name == @counter_name").sort_values('date')
data.plot(x='date', y="bike_count");

In [ ]:
X, y = data.drop(['bike_count', 'log_bike_count'], axis='columns'), data['log_bike_count']
X = X[['date']]

In [ ]:
n_samples_train = len(X) // 2  # temporal hold-out — NOT random, to respect causal ordering
X_train, y_train = X[:n_samples_train], y.iloc[:n_samples_train]
X_test, y_test = X[n_samples_train:], y.iloc[n_samples_train:]

## Direct prediction with Feature extraction

In [ ]:
def _encode_dates(X):
    X = X.copy()  # modify a copy of X
    # Encode the date information from the DateOfDeparture columns
    X.loc[:, 'year'] = X['date'].dt.year
    X.loc[:, 'month'] = X['date'].dt.month
    X.loc[:, 'day'] = X['date'].dt.day
    X.loc[:, 'weekday'] = X['date'].dt.day_of_week
    X.loc[:, 'hour'] = X['date'].dt.hour

    # Finally we can drop the original columns from the dataframe
    return X.drop(columns=["date"])

In [ ]:
df['date'].head()

In [ ]:
_encode_dates(df[['date']].head())

In [ ]:
from sklearn.preprocessing import FunctionTransformer

date_encoder = FunctionTransformer(_encode_dates, validate=False)
date_encoder.fit_transform(df[['date']]).head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

date_encoder = FunctionTransformer(_encode_dates)
regressor = Ridge()
selector = FunctionTransformer(
    lambda X: X[['weekday', 'hour']]
)
ohe = OneHotEncoder(handle_unknown='ignore')

clf_feature = make_pipeline(date_encoder, selector, ohe, regressor)
clf_feature.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import root_mean_squared_error

print(f'Train set, RMSE={root_mean_squared_error(y_train, clf_feature.predict(X_train)):.2f}')
print(f'Test set, RMSE={root_mean_squared_error(y_test, clf_feature.predict(X_test)):.2f}')

In [ ]:
def visualize_pred(y_pred, name="Ridge"):
    # Plot one normal week and the easter week
    start = pd.to_datetime("2021/03/29")
    end = start + pd.to_timedelta("7days")
    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharey=True)
    for i, ax in enumerate(axes):
        mask = ((X_test['date'] > start) & (X_test['date'] < end))
        df_viz = X_test.loc[mask].copy()
        df_viz['bike_count'] = np.exp(y_test[mask.values]) - 1
        df_viz['bike_count (predicted)'] = np.exp(y_pred[mask]) -  1
        
        df_viz.plot(x='date', y='bike_count', ax=ax)
        df_viz.plot(x='date', y='bike_count (predicted)', ax=ax, ls='--')
        ax.set_ylabel('bike_count')
        if i == 0:
            ax.set_title(f'Predictions with {name}')
        else:
            ax.set_xlabel("Easter week")
        start = end
        end = start + pd.to_timedelta("7days")

y_pred_features = clf_feature.predict(X_test)
visualize_pred(y_pred_features)

We can see the daily trend, and some of the week day differences are accounted for, however we still miss the details and the spikes in the morning and the evening are under-estimated.

Notice in the second plot the fact that Monday is the easter monday, a French bank holiday, which is completely missed by our model.

A useful way to visualize the error is to plot y_pred as a function of y_true,

In [ ]:
fig, ax = plt.subplots()
df_viz = pd.DataFrame({'y_true': y_test, 'y_pred': clf_feature.predict(X_test)})
df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.25, ax=ax)

Now let's do a proper cross-validation with `TimeSeriesSplit`:

> **Note on encoding:** `weekday` and `hour` are one-hot encoded here (not as plain integers, not as cyclic cos/sin). OHE works well for this problem because the model sees each hour/weekday as a distinct category with no implied ordering — Saturday and Sunday being adjacent is automatically captured. Cyclic encoding with cos/sin is an alternative worth trying; it uses fewer parameters but assumes the signal is smooth across the cycle boundary.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

cv = TimeSeriesSplit(n_splits=6)

# When using a scorer in scikit-learn it always needs to be better when smaller, hence the minus sign.
scores = cross_val_score(clf_feature, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
print('RMSE: ', -scores)
print(f'RMSE (all folds): {-scores.mean():.3} ± {(-scores).std():.3}')

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Modify the _encode_dates function to take into account the French holidays. And reevaluate the full pipeline above. Does it help significantly?</li>
    </ul>
</div>

Solution is in `solutions/bike-1_french-holidays.py`

In [ ]:
from pandas.tseries.holiday import AbstractHolidayCalendar, Holiday, EasterMonday, Easter
from pandas.tseries.offsets import Day, CustomBusinessDay

class FrBusinessCalendar(AbstractHolidayCalendar):
    """ Custom Holiday calendar for France based on
        https://en.wikipedia.org/wiki/Public_holidays_in_France
      - 1 January: New Year's Day
      - Moveable: Easter Monday (Monday after Easter Sunday)
      - 1 May: Labour Day
      - 8 May: Victory in Europe Day
      - Moveable Ascension Day (Thursday, 39 days after Easter Sunday)
      - 14 July: Bastille Day
      - 15 August: Assumption of Mary to Heaven
      - 1 November: All Saints' Day
      - 11 November: Armistice Day
      - 25 December: Christmas Day
    """
    rules = [
        Holiday('New Years Day', month=1, day=1),
        EasterMonday,
        Holiday('Labour Day', month=5, day=1),
        Holiday('Victory in Europe Day', month=5, day=8),
        Holiday('Ascension Day', month=1, day=1, offset=[Easter(), Day(39)]),
        Holiday('Bastille Day', month=7, day=14),
        Holiday('Assumption of Mary to Heaven', month=8, day=15),
        Holiday('All Saints Day', month=11, day=1),
        Holiday('Armistice Day', month=11, day=11),
        Holiday('Christmas Day', month=12, day=25)
    ]

cal = FrBusinessCalendar()

By adding this extra feature informing whether or not a day is a bank holiday, we obtain improved predictions, with a lower RMSE measured during CV.

<div class="alert alert-success">
    <b>EXERCISE</b>:
     <ul>
      <li>Add interaction terms between one-hot encoded hour and weekdays in your model using <code>sklearn.preprocessing.PolynomialFeatures</code>. Does it help?</li>
    </ul>
</div>

Solution is in `solutions/bike-2_polynomial-features.py`

## Patch-based approaches: the AR model

The feature-based models above make predictions **independently at each time step** — they know the calendar (weekday, hour) but not the recent bike counts. An AR model instead conditions on the *recent history of the signal itself*: given the last $p$ hourly counts, predict the next one. This naturally captures momentum (if it's busy now, it'll be busy in an hour).

The method is introduced in `04a-linear_forecasting_AR.ipynb`: the windows of the signal form a Hankel matrix, a regressor learns to predict the next value, and the `forecast` function below reinjects each prediction to predict the following one.

In [ ]:
from scipy.linalg import hankel


def forecast(model, x, n_samples, order=None):
    if order is None:
        order = model.coef_.shape[0]
    y_pred = np.empty(n_samples + order)  # the prediction
    y_pred[:order] = x[-order:]
    for i in range(n_samples):  # now predict successively one sample at a time
        y_pred[i + order] = model.predict(y_pred[i:i + order][None, :])[0]
    return y_pred[order:]

<div class="alert alert-success">
    <b>EXERCISE</b>:
    Compare the performance of an auto-regressive (AR) Ridge model with an AR model based on RandomForest
     <ol>
      <li>Form the training dataset <code>X_ar, y_ar</code> as hankel matrices from the signal we want to predict and the next time point.</li>
      <li>Train a <code>Ridge</code> model and a <code>RandomForestRegressor</code> with these training set.</li>
      <li>Predict the bike demand using the <code>forecast</code> function.</li>
      <li>Evalute the forecasted series with the <code>root_mean_square_error</code> and plot the forecast for the week of the 5th to the 12th April 2021.</li>
    </ol>
    
<i>Hint:</i> Use an AR model of order $p=167$ for a week.
    
</div>

Solution are in `solutions/bike-3_forecast-ridge.py` and `solutions/bike-3_forecast-rf.py`

We can perform a cross validation with the following code:


In [ ]:
p = 167  # AR order: 167 hours ≈ 1 week, enough to capture the weekly cycle

scores = []
for idx_train, idx_test in cv.split(X_train):
    X_ar, y_ar = hankel(y_train.iloc[idx_train].values, r=np.zeros(p))[:-p], y_train.iloc[idx_train].values[p:]
    model = Ridge().fit(X_ar, y_ar)
    y_forecast = forecast(model, y_ar, len(idx_test), p)
    scores.append(root_mean_squared_error(y_train.iloc[idx_test].values, y_forecast))

scores = np.array(scores)
print('RMSE: ', scores)
print(f'RMSE (all folds): {scores.mean():.3} ± {(scores).std():.3}')

# Using recent advances with Time-series Foundation Models

All the models above require **fitting on the target series**. Time-Series Foundation Models (TSFMs) flip this: they are pre-trained on millions of time series and can forecast **zero-shot** — no fitting on the bike data at all. This makes them easy to try as a baseline, and increasingly competitive with classical approaches.

As discussed with Tabular foundation models, recent trends is to replace learning a model with foundation models doing in context learning.
This trend has also started on Time-series forecasting with various Time-Series Foundation models (TSFM) which have been proposed in 2025.
Here we describe how to use 4 of them on our problem.

## Example 1 — TiRex (NX-AI)

TiRex is a compact, zero-shot forecasting foundation model based on xLSTM.
It accepts a history window and produces point and quantile forecasts

without training on the new dataset.

In [ ]:
# !pip install tirex-ts

In [ ]:
import torch
from tirex import load_model

# 1. Load the pre-trained TiRex model
# We use the base model "NX-AI/TiRex".
# If you don't have a GPU, it will fallback to CPU (may be slower for long contexts).
device = "cuda" if torch.cuda.is_available() else "cpu"
tirex_model = load_model("NX-AI/TiRex", device=device)

# 2. Prepare Data
# TiRex expects a torch tensor of shape (batch_size, sequence_length)
# We convert our context data (raw values) to a float32 tensor and add a batch dimension
context_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(0)
context_tensor = context_tensor.to(device)

# 3. Generate Forecast
# TiRex returns a tuple: (quantiles, mean_prediction)
# quantiles shape: (batch, prediction_length, num_quantiles)
# mean shape: (batch, prediction_length)
with torch.no_grad():
    _, forecast_tirex = tirex_model.forecast(
        context=context_tensor,
        prediction_length=len(y_test)
    )

# Extract the mean forecast for the first (and only) batch item and move to CPU
tirex_pred = forecast_tirex[0].cpu().numpy()
print("RMSE:", root_mean_squared_error(tirex_pred, y_test))
visualize_pred(tirex_pred, name="Tirex")

In [ ]:
scores_tirex = []
for idx_train, idx_test in cv.split(X_train):
    print(f" Scoring fold {len(scores_tirex) + 1}")
    context = torch.tensor(y_train.iloc[idx_train].values, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        _, forecast_tirex = tirex_model.forecast(
            context=context,
            prediction_length=len(idx_test)
        )
    y_forecast = forecast_tirex[0].cpu().numpy()
    scores_tirex.append(root_mean_squared_error(y_train.iloc[idx_test].values, y_forecast))

scores_tirex = np.array(scores_tirex)
print('RMSE: ', scores_tirex)
print(f'RMSE (all folds): {scores_tirex.mean():.3} ± {(scores_tirex).std():.3}')

## Example 2 -- Chronos (Amazon)

Chronos (based on T5) treats time-series values as a language of tokens. It is highly robust for univariate forecasting.

In [ ]:
# !pip install -U chronos-forecasting

In [ ]:
import torch
import numpy as np
from chronos import Chronos2Pipeline

# 1. Load the pre-trained Chronos model (using 'small' variant)
device = "cuda" if torch.cuda.is_available() else "cpu"
chronos_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map=device,
    dtype=torch.float32,
)

# 2. Prepare Data
# Chronos expects a tensor of shape (num_series, n_variates, sequence_length)
context_tensor = torch.tensor(
    y_train.values, dtype=torch.float32
)[None, None]

# 3. Generate Forecast
# pipeline.predict returns samples of shape (num_series, num_samples, prediction_length)
forecast_samples = chronos_pipeline.predict(context_tensor, len(y_test))

# Extract the median forecast as the point prediction
# forecast_samples is a list; index [0] gives the tensor for our single series
chronos_pred = torch.quantile(forecast_samples[0], 0.5, dim=1).squeeze(0).numpy()

print("RMSE:", root_mean_squared_error(chronos_pred, y_test))
visualize_pred(chronos_pred, name="Chronos2")

In [ ]:
scores_chronos = []
for idx_train, idx_test in cv.split(X_train):
    print(f" Scoring fold {len(scores_chronos) + 1} with Chronos...")
    context = torch.tensor(y_train.iloc[idx_train].values, dtype=torch.float32)[None, None]

    with torch.no_grad():
        forecast = chronos_pipeline.predict(context, len(idx_test))

    y_forecast = torch.quantile(forecast[0], 0.5, dim=1).squeeze(0).numpy()
    scores_chronos.append(root_mean_squared_error(y_train.iloc[idx_test].values, y_forecast))

scores_chronos = np.array(scores_chronos)
print('Chronos RMSE: ', scores_chronos)
print(f'RMSE (all folds): {scores_chronos.mean():.3} ± {(scores_chronos).std():.3}')

## Example 3 — TimesFM (Google)

TimesFM is a decoder-only foundation model from Google Research (ICML 2024), pre-trained on a large corpus of time series from Google's internal systems. Like Chronos, it runs **zero-shot** — no fitting on the bike data needed.

In [ ]:
# !pip install timesfm[torch]

In [ ]:
import timesfm
from huggingface_hub import hf_hub_download

# 1. Load and compile the model
# Workaround for timesfm v2.0.0 bug: from_pretrained leaks HF Hub kwargs into
# __init__. We also disable torch_compile (torch inductor version mismatch).
weights_path = hf_hub_download(
    repo_id="google/timesfm-2.5-200m-pytorch",
    filename=timesfm.TimesFM_2p5_200M_torch.WEIGHTS_FILENAME,
)
timesfm_model = timesfm.TimesFM_2p5_200M_torch(torch_compile=False)
# Load weights into the inner torch module (mirrors what _from_pretrained does)
timesfm_model.model.load_checkpoint(weights_path, torch_compile=False)

# max_horizon must cover the longest horizon we'll request:
# len(y_test) ~ 4954 hours and CV fold test sets can be similarly large.
timesfm_model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=8192,
        normalize_inputs=True,
    )
)

# 2. Prepare Data — TimesFM accepts a list of 1-D numpy arrays, one per series
context_array = y_train.values

# 3. Generate Forecast
# Returns (point_forecast, quantile_forecast)
# point_forecast shape: (n_series, horizon)
timesfm_pred_arr, _ = timesfm_model.forecast(
    horizon=len(y_test),
    inputs=[context_array],
)
timesfm_pred = timesfm_pred_arr[0]  # shape: (len(y_test),)
print("RMSE:", root_mean_squared_error(timesfm_pred, y_test))
# Plotting
visualize_pred(timesfm_pred, name="TimesFM")

In [ ]:
scores_timesfm = []
for idx_train, idx_test in cv.split(X_train):
    print(f" Scoring fold {len(scores_timesfm) + 1} with TimesFM...")
    context = y_train.iloc[idx_train].values
    timesfm_pred_arr, _ = timesfm_model.forecast(
        horizon=len(idx_test),
        inputs=[context],
    )
    y_forecast = timesfm_pred_arr[0]
    scores_timesfm.append(root_mean_squared_error(y_train.iloc[idx_test].values, y_forecast))

scores_timesfm = np.array(scores_timesfm)
print('TimesFM RMSE: ', scores_timesfm)
print(f'RMSE (all folds): {scores_timesfm.mean():.3} ± {(scores_timesfm).std():.3}')

## Example 4 — t0 (The Forecasting Company)

t0 is an open-weights foundation model released in 2026. Unlike the three models above, it can condition on
**known-future covariates**: features whose values are known over the forecast horizon, such as the calendar.
We can give it the hour, weekday and French holiday features that we engineered for the Ridge model,
and compare its forecast with and without them.

In [ ]:
# !pip install "tfc-t0[torch]>=0.6.0"

In [ ]:
from t0 import T0Forecaster

device = "cuda" if torch.cuda.is_available() else "cpu"
t0_model = T0Forecaster.from_pretrained("theforecastingcompany/t0-gamma").to(device).eval()


def calendar_covariates(dates):
    # Covariates are shaped (n_series, n_covariates, context + horizon)
    holidays = cal.holidays(dates.min(), dates.max())
    features = np.stack([
        dates.dt.hour, dates.dt.day_of_week, dates.dt.normalize().isin(holidays)
    ]).astype(np.float32)
    return features[None]


def t0_forecast(context, dates, covariates=True):
    # `dates` covers the context and the forecast horizon
    future_covariates = calendar_covariates(dates) if covariates else None
    with torch.no_grad():
        out = t0_model.predict(
            context, horizon=len(dates) - len(context), future_covariates=future_covariates
        )
    return out.median[0].cpu().numpy()


all_dates = pd.concat([X_train['date'], X_test['date']])
t0_pred = t0_forecast(y_train.values, all_dates, covariates=False)
t0_cov_pred = t0_forecast(y_train.values, all_dates)
print("RMSE without covariates:", root_mean_squared_error(t0_pred, y_test))
print("RMSE with covariates:", root_mean_squared_error(t0_cov_pred, y_test))
visualize_pred(t0_cov_pred, name="t0 with calendar covariates")

In [ ]:
scores_t0, scores_t0_cov = [], []
for idx_train, idx_test in cv.split(X_train):
    print(f" Scoring fold {len(scores_t0) + 1} with t0...")
    context = y_train.iloc[idx_train].values
    y_true = y_train.iloc[idx_test].values
    dates = X_train['date'].iloc[:idx_test[-1] + 1]
    scores_t0.append(root_mean_squared_error(y_true, t0_forecast(context, dates, covariates=False)))
    scores_t0_cov.append(root_mean_squared_error(y_true, t0_forecast(context, dates)))

scores_t0, scores_t0_cov = np.array(scores_t0), np.array(scores_t0_cov)
print(f't0 RMSE (all folds): {scores_t0.mean():.3} ± {scores_t0.std():.3}')
print(f't0 + covariates RMSE (all folds): {scores_t0_cov.mean():.3} ± {scores_t0_cov.std():.3}')

Compare the two scores with the forecast of the test set above. The calendar covariates help t0 when it forecasts
the 7 months of the test set, but they hurt it on the cross-validation folds, which each forecast about 4 weeks.
On a short horizon, the recent history already carries the daily and weekly cycles.
A feature that helps one model, or one horizon, can hurt another: measure it rather than assume it.

## Final comparison

All models evaluated with RMSLE (cross-validation on the training set, `TimeSeriesSplit`).

In [ ]:
import pandas as pd

# Collect CV scores computed above (all are RMSLE, lower is better)
ridge_cv = -cross_val_score(clf_feature, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')

results = pd.DataFrame({
    "Model": [
        "Ridge (weekday + hour)",
        "AR Ridge (p=167)",
        "TiRex (zero-shot)",
        "Chronos (zero-shot)",
        "TimesFM (zero-shot)",
        "t0 (zero-shot)",
        "t0 + calendar covariates",
    ],
    "Mean RMSLE": [
        ridge_cv.mean(),
        scores.mean(),        # AR Ridge scores computed above
        scores_tirex.mean(),
        scores_chronos.mean(),
        scores_timesfm.mean(),
        scores_t0.mean(),
        scores_t0_cov.mean(),
    ],
    "Std RMSLE": [
        ridge_cv.std(),
        scores.std(),
        scores_tirex.std(),
        scores_chronos.std(),
        scores_timesfm.std(),
        scores_t0.std(),
        scores_t0_cov.std(),
    ],
}).sort_values("Mean RMSLE")

results.style.format({"Mean RMSLE": "{:.3f}", "Std RMSLE": "{:.3f}"})

In [ ]:
mask = (X_test['date'] > '2021/03/29') & (X_test['date'] < '2021/04/05')
dates_mask = X_test.loc[mask, 'date']
actual = np.exp(y_test[mask.values]) - 1

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates_mask, actual, lw=2, label='Actual')
ax.plot(dates_mask, np.exp(clf_feature.predict(X_test[mask])) - 1, ls='--', label='Ridge')
ax.plot(dates_mask, np.exp(tirex_pred[mask.values]) - 1, ls='--', label='TiRex (zero-shot)')
ax.plot(dates_mask, np.exp(chronos_pred[mask.values]) - 1, ls='--', label='Chronos (zero-shot)')
ax.plot(dates_mask, np.exp(timesfm_pred[mask.values]) - 1, ls='--', label='TimesFM (zero-shot)')
ax.plot(dates_mask, np.exp(t0_cov_pred[mask.values]) - 1, ls='--', label='t0 + covariates')
ax.set_title('Model comparison — week of March 29, 2021')
ax.set_ylabel('bike_count')
plt.legend()
plt.tight_layout()
plt.show()